In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import v2

In [2]:
train_data = datasets.PCAM(
    root='data',
    split='train',
    download=False,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

In [11]:
train_data_sub = Subset(train_data, range(6400))
len(train_data_sub)

6400

In [12]:
train_data_sub[0][0].shape, train_data_sub[0][1]

(torch.Size([3, 96, 96]), 0)

In [13]:
batch_size = 64
data_t = DataLoader(train_data_sub, batch_size=batch_size, shuffle=True)

In [16]:
for X, y in data_t:
    print(X.shape, y.shape, y.dtype)
    break

torch.Size([64, 3, 96, 96]) torch.Size([64]) torch.int64


In [15]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
device

'mps'

In [47]:
class My_Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=(5,5), stride=1, padding=0),
            nn.BatchNorm2d(num_features=16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(5,5), stride=1, padding=0),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3,3), stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(in_features=64*9*9, out_features=1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(in_features=1024, out_features=64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        logits = self.layer_stack(x)
        return logits

In [48]:
model = My_Model().to(device)
print(model)

My_Model(
  (layer_stack): Sequential(
    (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (9): ReLU()
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Flatten(start_dim=1, end_dim=-1)
    (12): Linear(in_features=5184, out_features=1024, bias=True)
    (13): ReLU()
    (14): Dropout(p=0.5, inplace=False)
    (15): Linear(in_features=1024, out_features=64, bias=True)
    (16): ReLU()
    (17): Dropout(p=0.2, inplace=False)
    (18): Linear(in_f

In [49]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

In [52]:
def train(data_t, model, loss_fn, optimizer):
    size = len(data_t.dataset)
    print(size)
    model.train()
    for batch, (X, y) in enumerate(data_t):
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()

        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        

        if batch%20 == 0:
            loss = loss.item()
            print(f"loss = {loss:>7f}, [{batch}/{len(data_t)}]")

In [ ]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(data_t, model, loss_fn, optimizer)

Epoch 1
-------------------------------
6400
loss = 0.577122, [0/100]
loss = 0.542111, [20/100]
loss = 0.576489, [40/100]
loss = 0.518138, [60/100]
loss = 0.554605, [80/100]
Epoch 2
-------------------------------
6400
loss = 0.576604, [0/100]
loss = 0.515360, [20/100]
loss = 0.522026, [40/100]
loss = 0.558366, [60/100]
loss = 0.486670, [80/100]
Epoch 3
-------------------------------
6400
loss = 0.511919, [0/100]
loss = 0.529178, [20/100]
loss = 0.493082, [40/100]
loss = 0.507626, [60/100]
loss = 0.455562, [80/100]
Epoch 4
-------------------------------
6400
loss = 0.534485, [0/100]
loss = 0.553520, [20/100]
loss = 0.517488, [40/100]
loss = 0.550161, [60/100]
loss = 0.537223, [80/100]
Epoch 5
-------------------------------
6400
loss = 0.614703, [0/100]
loss = 0.544625, [20/100]
loss = 0.430771, [40/100]
loss = 0.467265, [60/100]
loss = 0.479191, [80/100]


In [66]:
from sklearn.metrics import roc_auc_score

test_data = datasets.PCAM(root='data', split='test', download=False,
                          transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]))

test_subset = Subset(test_data, range(128))

test_dl = DataLoader(test_subset, batch_size=batch_size)

def test(test_dl, model, loss_fn):
    size = len(test_dl)
    row_num = len(test_dl.dataset)
    print(size)
    test_loss, correct = 0.0, 0
    probs = []
    ys = []
    model.eval()
    with torch.no_grad():
        for X, y in test_dl:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            probs.append(torch.softmax(pred, 1)[:, 1].cpu())
            ys.append(y.cpu())

    test_loss /= size
    correct /= row_num
    auc = roc_auc_score(torch.cat(ys), torch.cat(probs))
    print(f"Test Metrics: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    print(auc)


In [67]:

test(test_dl, model, loss_fn)

2
Test Metrics: 
 Accuracy: 77.3%, Avg loss: 0.522954 

0.8301136363636363


In [62]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(data_t, model, loss_fn, optimizer)
    test(test_dl, model, loss_fn)

Epoch 1
-------------------------------
6400
loss = 0.516705, [0/100]
loss = 0.506891, [20/100]
loss = 0.516106, [40/100]
loss = 0.503410, [60/100]
loss = 0.344574, [80/100]
2
Test Metrics: 
 Accuracy: 75.8%, Avg loss: 0.524538 

Epoch 2
-------------------------------
6400
loss = 0.533617, [0/100]
loss = 0.321444, [20/100]
loss = 0.535780, [40/100]
loss = 0.503677, [60/100]
loss = 0.499389, [80/100]
2
Test Metrics: 
 Accuracy: 75.0%, Avg loss: 0.537665 

Epoch 3
-------------------------------
6400
loss = 0.501653, [0/100]
loss = 0.444107, [20/100]
loss = 0.524755, [40/100]
loss = 0.552286, [60/100]
loss = 0.459421, [80/100]
2
Test Metrics: 
 Accuracy: 76.6%, Avg loss: 0.474901 

Epoch 4
-------------------------------
6400
loss = 0.485202, [0/100]
loss = 0.510948, [20/100]
loss = 0.518837, [40/100]
loss = 0.483548, [60/100]
loss = 0.542797, [80/100]
2
Test Metrics: 
 Accuracy: 75.8%, Avg loss: 0.543589 

Epoch 5
-------------------------------
6400
loss = 0.592696, [0/100]
loss = 0.3

In [63]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


In [ ]:
model = My_Model().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

In [64]:
test(test_dl, model, loss_fn)

2
Test Metrics: 
 Accuracy: 77.3%, Avg loss: 0.522954 



In [ ]:
from sklearn.metrics import roc_auc_score
# inside eval loop:
probs.append(torch.softmax(pred, 1)[:, 1].cpu()); ys.append(y.cpu())
# after loop:
auc = roc_auc_score(torch.cat(ys), torch.cat(probs))